# 🚀 QLoRA Fine-Tuning on RunPod — Final

Fine-tune **Qwen2.5-1.5B-Instruct** with QLoRA, track with MLflow, compare runs, then **merge the winning adapter into the base model and push it to the Hugging Face Hub**.

**Run order:** top to bottom. Install cell first, then **restart the kernel once**, then continue.

**Hard-won environment lessons baked into this notebook:**
- The pod's pre-installed `torch` is matched to its GPU driver — **don't reinstall torch**. Overriding it caused a cascade of CUDA/driver/library mismatches.
- torch, transformers, tokenizers, and the training libs are **one matched set** — pinned together in a single install.
- `torchvision` isn't needed for text models and its version mismatch breaks `transformers` import — we remove it.
- Any cell that installs a package needs a **kernel restart** before the change is live.

> **RunPod note:** a pod *is* a container — no Docker/K8s inside. This notebook does GPU work only (train → merge → push).

## 1. Install dependencies (then RESTART THE KERNEL)

One atomic install with everything pinned to versions that agree with the pod's `torch 2.5.1+cu121`.
- `--ignore-installed blinker` steps around a Debian-managed package pip can't cleanly remove.
- We do **not** reinstall torch — the pod's build matches its driver.
- We remove `torchvision` (unused for LLMs; its mismatch breaks `transformers`).

After this runs: **Kernel → Restart**, then skip this cell and continue.

In [ ]:
# remove torchvision (not needed for text models; version mismatch breaks transformers import)
%pip uninstall -q -y torchvision

# one atomic, mutually-compatible install (torch stays as the pod shipped it)
%pip install -q --ignore-installed blinker \
    "transformers==4.46.3" \
    "tokenizers==0.20.3" \
    "accelerate==1.1.1" \
    "peft==0.13.2" \
    "trl==0.12.1" \
    "datasets==3.1.0" \
    "bitsandbytes==0.44.1" \
    "mlflow-skinny==2.17.2" \
    "huggingface_hub>=0.26.0" \
    "hf_transfer" \
    "sentencepiece>=0.2.0"

print("done — now Kernel → Restart, then continue from cell 2")

## 2. Environment check — GPU + compute dtype

The single most common QLoRA crash is requesting **bf16** on a GPU that doesn't support it.
- **bf16** needs Ampere+ (RTX 30xx/40xx, A4500, A100)
- **fp16** for older cards (T4)

We detect it once and reuse everywhere. We also set `expandable_segments` to avoid the
fragmentation OOM (an OOM where lots of memory is "reserved but unallocated").

In [ ]:
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # anti-fragmentation

import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("bf16 supported:", torch.cuda.is_bf16_supported())
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    print(
        "!! No CUDA. If you just installed, RESTART THE KERNEL. "
        "If torch was reinstalled, it may be built for a CUDA newer than the driver."
    )

if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    COMPUTE_DTYPE, DTYPE_NAME = torch.bfloat16, "bf16"
elif torch.cuda.is_available():
    COMPUTE_DTYPE, DTYPE_NAME = torch.float16, "fp16"
else:
    COMPUTE_DTYPE, DTYPE_NAME = torch.float32, "fp32-cpu"
print(">> compute dtype:", DTYPE_NAME)

In [ ]:
# sanity: confirm the whole stack imports cleanly (catches version mismatches early)
import mlflow
import torch
import transformers

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("all imports OK")

## 3. Configuration — every knob in one place

Run an experiment by changing values **here**, never in the logic below. Tuned for a
~20GB Ampere card (A4500). If you OOM, walk the ladder in the comment.

In [ ]:
CONFIG = {
    "experiment_name": "domainbot-qlora",
    "run_name": "baseline",  # change per experiment
    "base_model": "Qwen/Qwen2.5-1.5B-Instruct",
    "train_path": "data/processed/train.jsonl",
    "val_path": "data/processed/val.jsonl",
    "max_seq_length": 1024,
    # QLoRA quantization
    "load_in_4bit": True,
    "bnb_4bit_quant_type": "nf4",
    "bnb_4bit_use_double_quant": True,
    # LoRA adapter
    "lora_r": 16,
    "lora_alpha": 32,  # keep alpha = 2 * r
    "lora_dropout": 0.05,
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    # training
    "epochs": 3,
    "batch_size": 4,
    "grad_accum": 4,  # effective batch = 4*4 = 16
    "learning_rate": 2e-4,
    "warmup_ratio": 0.03,
    "weight_decay": 0.01,
    "gradient_checkpointing": True,
    "early_stopping_patience": 2,
    "seed": 42,
    "system_prompt": "You are DomainBot, a helpful and precise assistant. "
    "Answer accurately and concisely, and say when you don't know.",
}
SYSTEM = CONFIG["system_prompt"]
print("config ready — run_name:", CONFIG["run_name"])

# ---- OOM ladder (change ONE at a time, re-run) ----
# 1. max_seq_length 1024 -> 512     (biggest saver)
# 2. batch_size 4 -> 2  AND  grad_accum 4 -> 8   (keeps effective batch 16)
# 3. lora_r 16 -> 8
# memory ~= batch_size * max_seq_length * model_size

## 4. Get the data onto the pod

Upload your Day 1 `train.jsonl` / `val.jsonl` into `data/processed/` via the Jupyter file
browser, OR let the cell below write a tiny smoke-test set so the notebook runs end-to-end
immediately. It only writes if the files are missing — your real data is never overwritten.

In [ ]:
import json
import os

os.makedirs("data/processed", exist_ok=True)


def write_sample(path, rows):
    if os.path.exists(path):
        print("exists, keeping:", path)
        return
    with open(path, "w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")
    print("wrote sample:", path)


def ex(q, a):
    return {
        "messages": [
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": q},
            {"role": "assistant", "content": a},
        ]
    }


train_rows = [
    ex(
        "A train travels 120 km in 2 hours. What is its average speed?",
        "Average speed = distance / time = 120 km / 2 h = 60 km/h.",
    ),
    ex("What is the capital of Japan?", "The capital of Japan is Tokyo."),
    ex(
        "Rewrite in passive voice: 'The team finished the project.'",
        "The project was finished by the team.",
    ),
    ex(
        "What will the stock market do tomorrow?",
        "I can't predict future market movements — no one reliably can.",
    ),
    ex(
        "Summarize: solar and wind power costs have fallen sharply over the last decade.",
        "Solar and wind power have become much cheaper over the past decade.",
    ),
] * 6
val_rows = [
    ex("What is the capital of France?", "The capital of France is Paris."),
    ex("What is 15% of 200?", "15% of 200 is 30."),
]
write_sample(CONFIG["train_path"], train_rows)
write_sample(CONFIG["val_path"], val_rows)

## 5. Load the dataset + chat template

The `formatting_func` applies the model's **own chat template** — training format must match
inference exactly, or the fine-tune degrades into raw text completion.

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(CONFIG["base_model"])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

train_ds = load_dataset("json", data_files=CONFIG["train_path"], split="train")
val_ds = load_dataset("json", data_files=CONFIG["val_path"], split="train")
print("train:", len(train_ds), "| val:", len(val_ds))


def formatting_func(batch):
    return tokenizer.apply_chat_template(batch["messages"], tokenize=False)


print("\n--- one formatted example ---")
print(tokenizer.apply_chat_template(train_ds[0]["messages"], tokenize=False)[:400])

## 6. Load base model in 4-bit (the "Q" in QLoRA)

Frozen base loaded in 4-bit NF4 (~4x smaller); computed in bf16/fp16.

In [ ]:
from peft import prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=CONFIG["load_in_4bit"],
    bnb_4bit_quant_type=CONFIG["bnb_4bit_quant_type"],
    bnb_4bit_use_double_quant=CONFIG["bnb_4bit_use_double_quant"],
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)
model = AutoModelForCausalLM.from_pretrained(
    CONFIG["base_model"],
    quantization_config=bnb_config,
    torch_dtype=COMPUTE_DTYPE,
    device_map="auto",
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(
    model, use_gradient_checkpointing=CONFIG["gradient_checkpointing"]
)
print(">> base model loaded in 4-bit")

## 7. Attach LoRA adapters (the "LoRA" in QLoRA)

Freeze the base, add small trainable adapters. Printout should show **~1-3% trainable** —
that's QLoRA. If it says 100%, LoRA didn't attach.

In [ ]:
from peft import LoraConfig, get_peft_model

peft_config = LoraConfig(
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    target_modules=CONFIG["target_modules"],
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, peft_config)
trainable, total = model.get_nb_trainable_parameters()
print(f">> trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)  <- ~1-3%")

## 8. Train — with MLflow tracking

File-based MLflow (no server). Watch **eval_loss** each epoch: if it rises while train_loss
falls, that's **overfitting** — early stopping (patience 2) will halt and keep the best epoch.

In [ ]:
import os
import subprocess

os.environ["MLFLOW_TRACKING_URI"] = "file:outputs/mlruns"

from transformers import EarlyStoppingCallback
from trl import SFTConfig, SFTTrainer


def git_sha():
    try:
        return subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
    except Exception:
        return "nogit"


sft = SFTConfig(
    output_dir=f"outputs/adapters/{CONFIG['run_name']}",
    num_train_epochs=CONFIG["epochs"],
    per_device_train_batch_size=CONFIG["batch_size"],
    gradient_accumulation_steps=CONFIG["grad_accum"],
    learning_rate=CONFIG["learning_rate"],
    lr_scheduler_type="cosine",
    warmup_ratio=CONFIG["warmup_ratio"],
    weight_decay=CONFIG["weight_decay"],
    gradient_checkpointing=CONFIG["gradient_checkpointing"],
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    max_seq_length=CONFIG["max_seq_length"],
    bf16=(DTYPE_NAME == "bf16"),
    fp16=(DTYPE_NAME == "fp16"),
    seed=CONFIG["seed"],
    report_to=["mlflow"],
    run_name=CONFIG["run_name"],
)
trainer = SFTTrainer(
    model=model,
    args=sft,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    formatting_func=formatting_func,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=CONFIG["early_stopping_patience"])],
)

mlflow.set_experiment(CONFIG["experiment_name"])
with mlflow.start_run(run_name=CONFIG["run_name"]):
    mlflow.log_params(
        {
            "base_model": CONFIG["base_model"],
            "lora_r": CONFIG["lora_r"],
            "lora_alpha": CONFIG["lora_alpha"],
            "epochs": CONFIG["epochs"],
            "learning_rate": CONFIG["learning_rate"],
            "compute_dtype": DTYPE_NAME,
            "trainable_pct": round(100 * trainable / total, 3),
            "git_sha": git_sha(),
            "seed": CONFIG["seed"],
        }
    )
    result = trainer.train()
    metrics = trainer.evaluate()
    mlflow.log_metrics(
        {
            "train_loss": result.training_loss,
            "eval_loss": metrics.get("eval_loss", float("nan")),
        }
    )
    print(
        f"\n>> done. train_loss={result.training_loss:.4f} eval_loss={metrics.get('eval_loss'):.4f}"
    )

## 9. Save the adapter (few MB) + this run's generations

Saves the adapter AND writes `sample_generations.json` so you can compare runs later without
reloading. During experiments we save only adapters — never merged models.

In [ ]:
import torch

adapter_dir = f"outputs/adapters/{CONFIG['run_name']}/final_adapter"
trainer.model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print(">> adapter saved to:", adapter_dir)


def generate(prompt, max_new_tokens=150):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(trainer.model.device)
    with torch.no_grad():
        out = trainer.model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )
    return tokenizer.decode(
        out[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True
    ).strip()


prompts = [
    "A train travels 150 km in 3 hours. What is its average speed?",
    "What is the capital of Japan?",
    "Rewrite in passive voice: 'The team finished the project.'",
    "What will the stock market do tomorrow?",
    "Summarize in one sentence: solar and wind power costs have fallen sharply over the past decade.",
]
samples = [{"prompt": p, "response": generate(p)} for p in prompts]
with open(f"outputs/adapters/{CONFIG['run_name']}/sample_generations.json", "w") as f:
    json.dump(samples, f, indent=2, ensure_ascii=False)

for s in samples[:3]:
    print("Q:", s["prompt"][:60])
    print("A:", s["response"][:140], "\n")

## 🔁 Run more experiments

To run another experiment, edit CONFIG and **re-run cells 6–9** (base → LoRA → train → save).
Change ONE thing at a time so comparisons are fair.

```python
# more epochs (start from BASE — never continue from a trained model):
CONFIG["run_name"] = "more_epochs"; CONFIG["epochs"] = 5; CONFIG["learning_rate"] = 1e-4

# higher rank (keep alpha = 2*r):
CONFIG["run_name"] = "rank32"; CONFIG["lora_r"] = 32; CONFIG["lora_alpha"] = 64
#   then RESET epochs/lr back to baseline (3 / 2e-4) so only rank differs
```
Each run writes its own `outputs/adapters/<run_name>/` folder — no deletion needed.

## 10. Compare all runs — numbers + generations

Two signals. **eval_loss** (which run predicts best) AND **generations** (which run actually
answers well). eval_loss alone is not enough — runs can tie on loss but differ on correctness.

In [ ]:
import mlflow

mlflow.set_tracking_uri("file:outputs/mlruns")
exp = mlflow.get_experiment_by_name(CONFIG["experiment_name"])
df = mlflow.search_runs(experiment_ids=[exp.experiment_id])
cols = {
    "tags.mlflow.runName": "run",
    "params.lora_r": "r",
    "params.epochs": "epochs",
    "params.learning_rate": "lr",
    "metrics.eval_loss": "eval_loss",
}
have = [c for c in cols if c in df.columns]
view = df[have].rename(columns=cols).dropna(subset=["eval_loss"]).sort_values("eval_loss")
print(view.to_string(index=False))

In [ ]:
# read the saved generations side by side (the tiebreaker)
import json
import os

for run in ["baseline", "more_epochs", "rank32"]:
    p = f"outputs/adapters/{run}/sample_generations.json"
    if not os.path.exists(p):
        continue
    print(f"\n{'='*60}\n{run}\n{'='*60}")
    for s in json.load(open(p)):
        print("Q:", s["prompt"][:60])
        print("A:", s["response"][:160], "\n")

### How to pick the winner

1. **Correctness first** — does it get facts (capital of Japan) and reasoning (50 km/h) right?
   A run that's confidently wrong loses even with lower eval_loss.
2. **Behavior** — does the honesty prompt admit uncertainty instead of hallucinating?
3. **Tiebreak on simplicity** — if two are equal, pick the cheaper (lower r, fewer epochs).

Set `WINNER` below to the run you chose.

## 11. Merge the WINNER (base + adapter) → full model

Merge into the model at **full precision** (not 4-bit — merging into 4-bit is lossy). We free
the training model, reload the base in bf16/fp16, attach the winning adapter, and `merge_and_unload()`.

In [ ]:
WINNER = "rank32"  # <-- set to your chosen run

import gc

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM

# free the 4-bit training model to make room for the full-precision merge
try:
    del model, trainer
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

adapter_dir = f"outputs/adapters/{WINNER}/final_adapter"
base_fp = AutoModelForCausalLM.from_pretrained(
    CONFIG["base_model"],
    torch_dtype=COMPUTE_DTYPE,
    device_map="auto",
)
merged = PeftModel.from_pretrained(base_fp, adapter_dir).merge_and_unload()

merged_dir = f"outputs/merged/{WINNER}"
merged.save_pretrained(merged_dir, safe_serialization=True)
tokenizer.save_pretrained(merged_dir)
print(">> merged model saved to:", merged_dir)

# sanity check the merged model
_m = merged.eval()
_msgs = [
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": "What is the capital of Japan?"},
]
_t = tokenizer.apply_chat_template(_msgs, tokenize=False, add_generation_prompt=True)
_i = tokenizer(_t, return_tensors="pt").to(_m.device)
with torch.no_grad():
    _o = _m.generate(
        **_i,
        max_new_tokens=40,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
    )
print(
    "merged says:",
    tokenizer.decode(_o[0][_i["input_ids"].shape[1] :], skip_special_tokens=True).strip(),
)

## 12. Push the merged model to the Hugging Face Hub

The Hub is your **model registry** — Day 4 serving pulls weights from here (never bake weights
into a Docker image). Needs a token with **write** access:
https://huggingface.co/settings/tokens

In [ ]:
from huggingface_hub import login, whoami

login()  # uses HF_TOKEN env if set, else prompts
HF_USERNAME = whoami()["name"]
print("logged in as:", HF_USERNAME)

In [ ]:
from huggingface_hub import create_repo, upload_folder

REPO_ID = f"{HF_USERNAME}/domainbot-1.5b-{WINNER}"  # rename if you like
create_repo(REPO_ID, repo_type="model", private=True, exist_ok=True)

card = f"""---
license: apache-2.0
base_model: {CONFIG["base_model"]}
tags: [qlora, peft, fine-tuned, domainbot]
---

# DomainBot 1.5B ({WINNER})

QLoRA fine-tune of `{CONFIG["base_model"]}` (merged, ready to serve with vLLM).

- LoRA r={CONFIG["lora_r"]}, alpha={CONFIG["lora_alpha"]}
- selected as winner by eval_loss + generation quality across 3 runs

System prompt used in training:
> {SYSTEM}
"""
with open(f"outputs/merged/{WINNER}/README.md", "w") as f:
    f.write(card)

upload_folder(
    repo_id=REPO_ID,
    folder_path=f"outputs/merged/{WINNER}",
    repo_type="model",
    commit_message=f"merged winner: {WINNER}",
)
print(f"\n>> pushed: https://huggingface.co/{REPO_ID}")

In [ ]:
# verify the model files actually landed on the Hub before terminating the pod
from huggingface_hub import list_repo_files

files = list_repo_files(REPO_ID)
print(files)
assert any(f.endswith(".safetensors") for f in files), "no weights on the Hub!"
print(">> verified: weights present on the Hub")

## 13. Preserve, then terminate

The pod is disposable. Save what you can't regenerate, then **terminate** (a stopped pod still
bills storage).

In [ ]:
# freeze the hard-won environment so future pods install in ONE line
%pip freeze > requirements-lock.txt
print("wrote requirements-lock.txt")
print("Download via the file browser: requirements-lock.txt, outputs/mlruns/")
print("Then TERMINATE the pod and update COSTS.md with GPU hours.")

## ✅ Done

You have:
- 3 tracked runs in `outputs/mlruns` (compare locally with `mlflow ui`)
- 3 lightweight adapters + saved generations
- the **merged winner** on the HF Hub, ready for Day 4 serving
- `requirements-lock.txt` — reproducible environment

**Next (Day 3):** put this model behind the golden-fact **gate** (`validate_model.py`) and
register it as a versioned model before it's allowed to deploy.

> **Model repo vs Space:** we pushed to a **model repo** (weights for serving — what vLLM pulls).
> A **Space** is a hosted demo app; make one separately if you want a clickable UI, pointing it
> at this model repo.